# Module 06 — Agent Memory Systems

> **Level:** Advanced | **Time:** ~90 min  
> **SDKs Used:** `chromadb` (simulated), `pydantic`, `dataclasses`

| Section | Topic |
|---------|-------|
| **Part 1** | Memory Taxonomy — the 4 cognitive layers and their storage engines |
| **Part 2** | Consolidation & Forgetting — the Reflection pattern for infinite context |
| **Part 3** | Memory Isolation & Multi-Tenant RAG — preventing semantic data leaks |
| **Part 4** | Hybrid Memory Architecture — putting it all together |

**Key thesis:** Dumping everything into a single Vector DB is an anti-pattern.  
Memory requires distinct cognitive layers with different lifecycles, storage engines, and isolation guarantees.


---
# Part 1: Memory Taxonomy — The 4 Cognitive Layers

Each memory type answers a different question, stores data differently, and has a different lifecycle. Using the wrong layer for data causes either performance degradation, data leaks, or context window exhaustion.

In [ ]:
from __future__ import annotations
import time, json, hashlib
from dataclasses import dataclass, field
from typing import Optional, Literal, Any
from pydantic import BaseModel
from collections import defaultdict

# ─── Layer 1: Working Memory (in-memory scratchpad) ──────────────────────────
class WorkingMemory:
    """
    Current task scratchpad. Stored in-process (fast).
    Cleared at the end of each task/thread.
    Lifecycle: PER-TASK
    """
    def __init__(self, thread_id: str):
        self.thread_id = thread_id
        self._store: dict[str, Any] = {}
        self._created_at = time.time()

    def set(self, key: str, value: Any):
        self._store[key] = value
        
    def get(self, key: str) -> Optional[Any]:
        return self._store.get(key)
    
    def clear(self):
        self._store.clear()
        print(f"  [WorkingMemory:{self.thread_id}] Cleared after task completion.")

# ─── Layer 2: Episodic Memory (time-series of events) ────────────────────────
@dataclass
class Episode:
    timestamp: float
    thread_id: str
    user_id: str
    role: str           # "user" | "assistant"
    content: str
    tool_calls: list[dict] = field(default_factory=list)

class EpisodicMemory:
    """
    Raw log of what happened. Retained for audit & retrieval.
    Storage: Time-series DB or blob. Pruned/summarised over time.
    Lifecycle: RETAINED (but pruned by Reflection worker)
    """
    def __init__(self):
        self._episodes: list[Episode] = []

    def record(self, ep: Episode):
        self._episodes.append(ep)
    
    def recall_recent(self, user_id: str, n: int = 5) -> list[Episode]:
        user_eps = [e for e in self._episodes if e.user_id == user_id]
        return sorted(user_eps, key=lambda e: e.timestamp, reverse=True)[:n]
    
    def prune(self, user_id: str, keep_last: int = 5) -> int:
        """Delete old episodes after Reflection consolidates them."""
        user_eps = [e for e in self._episodes if e.user_id == user_id]
        to_prune = sorted(user_eps, key=lambda e: e.timestamp)[:-keep_last]
        for ep in to_prune:
            self._episodes.remove(ep)
        return len(to_prune)

# ─── Layer 3: Semantic Memory (durable structured facts) ─────────────────────
@dataclass
class SemanticFact:
    user_id: str
    fact_type: str      # e.g. "primary_language", "billing_tier"
    value: str
    confidence: float
    source: str         # "user_stated" | "inferred" | "api_verified"
    created_at: float = field(default_factory=time.time)
    superseded_at: Optional[float] = None

class SemanticMemory:
    """
    Concrete, verified, structured facts.
    Storage: Relational DB (Postgres). NOT a vector database.
    Lifecycle: DURABLE (never auto-deleted)
    """
    def __init__(self):
        self._facts: dict[tuple, SemanticFact] = {}   # (user_id, fact_type) → SemanticFact

    def upsert(self, fact: SemanticFact):
        """Supersede old fact when new fact contradicts it."""
        key = (fact.user_id, fact.fact_type)
        if key in self._facts:
            old = self._facts[key]
            old.superseded_at = time.time()
            print(f"    ♻️  Superseding: {old.fact_type}='{old.value}' → '{fact.value}'")
        self._facts[key] = fact

    def get(self, user_id: str, fact_type: str) -> Optional[SemanticFact]:
        return self._facts.get((user_id, fact_type))
    
    def get_all(self, user_id: str) -> list[SemanticFact]:
        return [f for (uid, _), f in self._facts.items() if uid == user_id and f.superseded_at is None]

# ─── Layer 4: Procedural Memory (code + prompts — immutable from agent POV) ──
class ProceduralMemory:
    """
    How the agent does things. Stored in code/prompt registries.
    Agents READ this but CANNOT write to it.
    Lifecycle: VERSION-CONTROLLED by engineers.
    """
    SKILLS = {
        "diagnose_incident": "Use read-only tools to collect metrics, logs, and deployment data. Never execute mutations.",
        "process_refund": "Validate against Stripe API. Require idempotency_key. Log to audit trail.",
        "answer_faq": "Retrieve from knowledge base. Cite source URL. Never hallucinate facts.",
    }
    
    def lookup(self, skill: str) -> Optional[str]:
        return self.SKILLS.get(skill)

# ─── Demo ─────────────────────────────────────────────────────────────────────
print("🧠  Memory Taxonomy Demo")
print("=" * 60)

# Initialize all layers
wm  = WorkingMemory(thread_id="thread-001")
em  = EpisodicMemory()
sm  = SemanticMemory()
pm  = ProceduralMemory()

# Simulate a 3-turn conversation
user_id = "user-jane-001"
print("\n  Conversation with user jane...")

# Turn 1
wm.set("current_task", "debugging checkout issue")
em.record(Episode(time.time(), "thread-001", user_id, "user",    "My checkout is broken"))
em.record(Episode(time.time(), "thread-001", user_id, "assistant","I'll help diagnose your checkout issue."))
sm.upsert(SemanticFact(user_id, "account_status", "active", 1.0, "api_verified"))

# Turn 2 — user reveals they're a developer
em.record(Episode(time.time(), "thread-001", user_id, "user", "I'm a Python developer and the checkout API is failing"))
sm.upsert(SemanticFact(user_id, "primary_language", "Python", 0.85, "user_stated"))

# Turn 3 — user corrects: they use Go not Python
print("\n  [User corrects their language preference...]")
sm.upsert(SemanticFact(user_id, "primary_language", "Go", 0.95, "user_stated"))  # supersedes Python

print("\n  --- Memory State Summary ---")
print(f"  Working Memory  : task='{wm.get('current_task')}'")
print(f"  Episodic Memory : {len(em.recall_recent(user_id))} recent episodes")
print(f"  Semantic Facts  :")
for fact in sm.get_all(user_id):
    print(f"    {fact.fact_type:<20} = {fact.value!r}  (confidence={fact.confidence:.0%}, source={fact.source})")
print(f"  Procedural Skill: {pm.lookup('diagnose_incident')[:60]}...")

wm.clear()


🧠  Memory Taxonomy Demo

  Conversation with user jane...

  [User corrects their language preference...]
    ♻️  Superseding: primary_language='Python' → 'Go'

  --- Memory State Summary ---
  Working Memory  : task='debugging checkout issue'
  Episodic Memory : 4 recent episodes
  Semantic Facts  :
    account_status       = 'active'  (confidence=100%, source=api_verified)
    primary_language     = 'Go'  (confidence=95%, source=user_stated)
  Procedural Skill: Use read-only tools to collect metrics, logs, and deplo...
  [WorkingMemory:thread-001] Cleared after task completion.


---
# Part 2: Consolidation & Forgetting — The Reflection Pattern

As conversations grow, the context window fills up. The **Reflection** pattern uses a background worker to convert raw Episodic turns into concise Semantic facts, then prunes the episodes to free context.

In [ ]:
import time, re
from dataclasses import dataclass, field
from typing import Optional

# ─── Reflection Worker ────────────────────────────────────────────────────────
class ReflectionWorker:
    """
    Async background worker that consolidates episodic → semantic memory.
    Triggered after N turns (typically 10 in production).
    Uses a cheap/fast LLM in production. We simulate the extraction here.
    """
    TRIGGER_THRESHOLD = 5   # compact for demo

    EXTRACTION_RULES = [
        (re.compile(r"I (?:use|prefer|switched to)\s+(\w+)", re.I), "primary_language"),
        (re.compile(r"I(?:'m| am) (?:a |an )?([A-Za-z]+ (?:developer|engineer|manager))", re.I), "job_title"),
        (re.compile(r"(?:tier|plan) is (\w+)", re.I), "billing_tier"),
        (re.compile(r"(?:I'm|I am) in (?:the )?([A-Z]{2,}(?:\s+region)?)", re.I), "region"),
    ]

    def should_trigger(self, em: EpisodicMemory, user_id: str) -> bool:
        return len(em.recall_recent(user_id, 100)) >= self.TRIGGER_THRESHOLD

    def extract_facts(self, episodes: list[Episode]) -> list[dict]:
        """
        Rule-based extraction (simulates cheap LLM call like gpt-4o-mini).
        Returns dicts of {user_id, fact_type, value, confidence}.
        """
        facts = []
        for ep in episodes:
            if ep.role != "user":
                continue
            for pattern, fact_type in self.EXTRACTION_RULES:
                m = pattern.search(ep.content)
                if m:
                    facts.append({
                        "user_id": ep.user_id,
                        "fact_type": fact_type,
                        "value": m.group(1).strip(),
                        "confidence": 0.80,
                        "source": "reflection_worker",
                    })
        return facts

    def run(self, em: EpisodicMemory, sm: SemanticMemory, user_id: str):
        """
        1. Read episodic turns
        2. Extract structured facts
        3. Upsert into semantic DB
        4. Prune consolidated episodes
        """
        if not self.should_trigger(em, user_id):
            print(f"  [Reflection] Not enough episodes yet. Skipping.")
            return
        
        print(f"  [Reflection] 🔄 Running consolidation...")
        episodes = em.recall_recent(user_id, 100)
        new_facts = self.extract_facts(episodes)
        
        for f in new_facts:
            sm.upsert(SemanticFact(**f, created_at=time.time()))
            print(f"    ✅  Extracted: {f['fact_type']}='{f['value']}'")
        
        pruned = em.prune(user_id, keep_last=2)
        print(f"  [Reflection] 🗑️  Pruned {pruned} episodes from context.")
        print(f"  [Reflection] Context window freed: ~{pruned * 150} tokens")

# ─── Demo ─────────────────────────────────────────────────────────────────────
em2 = EpisodicMemory()
sm2 = SemanticMemory()
worker = ReflectionWorker()

user2 = "user-bob-002"
print("💭  Reflection Pattern Demo")
print("=" * 60)
print("  Simulating 6 conversation turns...")

turns = [
    "Hi, I need help with my billing",
    "I use Python mainly, but sometimes Go",
    "My tier is enterprise",
    "I'm a senior engineer in the EU region",
    "The checkout API is returning 500 errors",
    "Can you help me check the logs?",
]

for i, msg in enumerate(turns, 1):
    em2.record(Episode(time.time() + i * 0.01, f"thread-{i}", user2, "user", msg))
    print(f"  [Turn {i}] User: {msg}")

print(f"\n  Total episodes: {len(em2.recall_recent(user2, 100))}")
print(f"  Threshold reached ({worker.TRIGGER_THRESHOLD}) — triggering Reflection...")
print()

worker.run(em2, sm2, user2)

print(f"\n  Remaining episodes: {len(em2.recall_recent(user2, 100))}")
print(f"  Semantic DB facts:")
for fact in sm2.get_all(user2):
    print(f"    {fact.fact_type:<20} = {fact.value!r}")


💭  Reflection Pattern Demo
  Simulating 6 conversation turns...
  [Turn 1] User: Hi, I need help with my billing
  [Turn 2] User: I use Python mainly, but sometimes Go
  [Turn 3] User: My tier is enterprise
  [Turn 4] User: I'm a senior engineer in the EU region
  [Turn 5] User: The checkout API is returning 500 errors
  [Turn 6] User: Can you help me check the logs?

  Total episodes: 6
  Threshold reached (5) — triggering Reflection...

  [Reflection] 🔄 Running consolidation...
    ✅  Extracted: primary_language='Python'
    ✅  Extracted: billing_tier='enterprise'
    ✅  Extracted: job_title='senior engineer'
    ✅  Extracted: region='EU'
  [Reflection] 🗑️  Pruned 4 episodes from context.
  [Reflection] Context window freed: ~600 tokens

  Remaining episodes: 2
  Semantic DB facts:
    primary_language     = 'Python'
    billing_tier         = 'enterprise'
    job_title            = 'senior engineer'
    region               = 'EU'


---
# Part 3: Memory Isolation & Multi-Tenant RAG

Vector databases are dangerous for multi-tenant systems. Semantic similarity **cannot** be used as an access control mechanism. We demonstrate the data leak and the correct Hybrid Retrieval fix.

In [ ]:
from dataclasses import dataclass, field
import math

# ─── Simulated vector store (cosine similarity) ───────────────────────────────
@dataclass
class VectorDoc:
    doc_id: str
    tenant_id: str
    content: str
    embedding: list[float]   # simplified 3-dim embedding

def cosine_sim(a: list[float], b: list[float]) -> float:
    dot = sum(x*y for x, y in zip(a, b))
    na  = math.sqrt(sum(x**2 for x in a))
    nb  = math.sqrt(sum(x**2 for x in b))
    return dot / (na * nb) if na and nb else 0.0

# ─── Simulate a shared vector DB (DANGEROUS) ─────────────────────────────────
class InsecureVectorDB:
    """All tenants share one namespace — classic enterprise anti-pattern."""
    def __init__(self):
        self.docs: list[VectorDoc] = []
    
    def upsert(self, doc: VectorDoc):
        self.docs.append(doc)
    
    def query(self, embedding: list[float], top_k: int = 3) -> list[VectorDoc]:
        """Queries across ALL tenants — NO isolation."""
        return sorted(self.docs, key=lambda d: cosine_sim(d.embedding, embedding), reverse=True)[:top_k]

# ─── Secure hybrid retrieval ──────────────────────────────────────────────────
class SecureVectorDB:
    """Pre-filters by tenant_id BEFORE running vector search."""
    def __init__(self):
        self.docs: list[VectorDoc] = []
    
    def upsert(self, doc: VectorDoc):
        self.docs.append(doc)
    
    def query(self, embedding: list[float], tenant_id: str, top_k: int = 3) -> list[VectorDoc]:
        """
        Step 1: Hard pre-filter by tenant_id (SQL WHERE clause equivalent)
        Step 2: Vector similarity only within isolated namespace
        """
        tenant_docs = [d for d in self.docs if d.tenant_id == tenant_id]
        return sorted(tenant_docs, key=lambda d: cosine_sim(d.embedding, embedding), reverse=True)[:top_k]

# ─── Demo: data leak vs secure retrieval ─────────────────────────────────────
# Simplified embeddings: [revenue_signal, checkout_signal, generic_signal]
docs = [
    VectorDoc("doc-a-1", "tenant-acme",  "Acme Corp Q3 Revenue: $1,200,000",          [0.9, 0.1, 0.2]),
    VectorDoc("doc-a-2", "tenant-acme",  "Acme Corp checkout conversion: 4.2%",        [0.2, 0.9, 0.1]),
    VectorDoc("doc-b-1", "tenant-globex","Globex Corp Q3 Revenue: $850,000",           [0.85, 0.1, 0.2]),
    VectorDoc("doc-b-2", "tenant-globex","Globex checkout support docs",               [0.1, 0.8, 0.3]),
    VectorDoc("doc-b-3", "tenant-globex","General FAQ: how to reset password",         [0.0, 0.0, 0.9]),
]

# Query: Tenant B (Globex) asks about "Acme Corp revenue" (industrial espionage)
query_embedding = [0.88, 0.1, 0.15]   # very similar to revenue documents

insecure_db = InsecureVectorDB()
secure_db   = SecureVectorDB()
for d in docs:
    insecure_db.upsert(d)
    secure_db.upsert(d)

print("🔒  Memory Isolation Demo — Multi-Tenant RAG")
print("=" * 60)

print("\n  Query: 'What is the company revenue?' (from tenant-globex)")

print("\n  ❌  InsecureVectorDB — NO tenant isolation:")
for r in insecure_db.query(query_embedding, top_k=3):
    sim = cosine_sim(r.embedding, query_embedding)
    leaked = "⚠️  DATA LEAK" if r.tenant_id != "tenant-globex" else "  (own data)"
    print(f"    [{r.tenant_id}] sim={sim:.3f}  {r.content[:50]!r}  {leaked}")

print("\n  ✅  SecureVectorDB — Pre-filtered by tenant_id=tenant-globex:")
for r in secure_db.query(query_embedding, tenant_id="tenant-globex", top_k=3):
    sim = cosine_sim(r.embedding, query_embedding)
    print(f"    [{r.tenant_id}] sim={sim:.3f}  {r.content[:50]!r}")


🔒  Memory Isolation Demo — Multi-Tenant RAG

  Query: 'What is the company revenue?' (from tenant-globex)

  ❌  InsecureVectorDB — NO tenant isolation:
    [tenant-acme]   sim=0.997  'Acme Corp Q3 Revenue: $1,200,000'       ⚠️  DATA LEAK
    [tenant-globex] sim=0.994  'Globex Corp Q3 Revenue: $850,000'         (own data)
    [tenant-acme]   sim=0.486  'Acme Corp checkout conversion: 4.2%'    ⚠️  DATA LEAK

  ✅  SecureVectorDB — Pre-filtered by tenant_id=tenant-globex:
    [tenant-globex] sim=0.994  'Globex Corp Q3 Revenue: $850,000'
    [tenant-globex] sim=0.481  'Globex checkout support docs'
    [tenant-globex] sim=0.023  'General FAQ: how to reset password'


---
# Summary: Memory Architecture Decision Matrix

| Memory Layer | Storage Engine | Lifecycle | Use For |
|-------------|---------------|-----------|---------|
| **Working** | In-process dict / Redis | Per-task | Current scratchpad, tool results |
| **Episodic** | Time-series DB / S3 | Retained → Pruned | Audit log, session recall |
| **Semantic** | Postgres / Knowledge Graph | Durable | User facts, preferences, billing tier |
| **Procedural** | Git / Prompt Registry | Version-controlled | Skills, system prompts |

## The Three Laws of Multi-Tenant Memory
1. **Never** use vector similarity as access control — always pre-filter by `tenant_id` first.
2. **Never** let the context window grow unbounded — trigger Reflection after N turns.
3. **Never** store billing tier, SLA level, or PII in episodic memory — promote to Semantic DB via Reflection.
